# Homework 7 — Role-Aware LangGraph Orchestration

This notebook extends the existing Health Psychology RAG project with a framework-based orchestration layer built with LangGraph.

The workflow routes user requests between:

- Health Psychology RAG;
- product analytics;
- access-denied handling for unauthorized analytics requests;
- clarification for unsupported or unclear requests.

A key part of the workflow is role-aware routing:

- an authenticated admin can access product analytics;
- an external user cannot access product analytics;
- the same analytics request therefore follows different graph paths depending on trusted user state.

The goal of this homework is to represent existing custom workflow logic using explicit:

- state;
- nodes;
- edges;
- conditional routing;
- execution traces.

The implementation intentionally keeps the underlying tools simple so that the LangGraph orchestration remains visible and easy to inspect.

## Stage 1 — Connect the repository and prepare the environment

In [1]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess


GITHUB_USER = "swanksenia"
REPO_NAME = "health-psychology-rag-kb"

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
PROJECT_ROOT = Path("/content") / REPO_NAME


github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise ValueError(
        "GITHUB_TOKEN was not found in Colab Secrets."
    )


# Temporary helper for secure GitHub authentication.
askpass_path = Path("/content/git_askpass.sh")

askpass_path.write_text(
    """#!/bin/sh
case "$1" in
    *Username*) echo "x-access-token" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
""",
    encoding="utf-8",
)

askpass_path.chmod(0o700)


git_environment = os.environ.copy()
git_environment["GITHUB_TOKEN"] = github_token
git_environment["GIT_ASKPASS"] = str(askpass_path)
git_environment["GIT_TERMINAL_PROMPT"] = "0"


if (PROJECT_ROOT / ".git").exists():
    print("Repository already exists. Pulling latest changes...")

    subprocess.run(
        [
            "git",
            "-C",
            str(PROJECT_ROOT),
            "pull",
            "--ff-only",
        ],
        check=True,
        env=git_environment,
    )
else:
    print("Cloning repository...")

    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            str(PROJECT_ROOT),
        ],
        check=True,
        env=git_environment,
    )


print("Project root:", PROJECT_ROOT)

Cloning repository...
Project root: /content/health-psychology-rag-kb


In [2]:
subprocess.run(
    [
        "git",
        "-C",
        str(PROJECT_ROOT),
        "config",
        "user.name",
        "Kseniia Lebedieva",
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        str(PROJECT_ROOT),
        "config",
        "user.email",
        "lebedevaky@gmail.com",
    ],
    check=True,
)

print("Git identity configured.")

Git identity configured.


In [3]:
def run_git_command(arguments):
    result = subprocess.run(
        [
            "git",
            "-C",
            str(PROJECT_ROOT),
            *arguments,
        ],
        check=True,
        capture_output=True,
        text=True,
    )

    return result.stdout.strip()


current_branch = run_git_command(
    ["branch", "--show-current"]
)

latest_commit = run_git_command(
    ["log", "-1", "--oneline"]
)

git_status = run_git_command(
    ["status", "--short"]
)


print("Branch:", current_branch)
print("Latest commit:", latest_commit)

print(
    "Repository status:",
    git_status if git_status else "Working tree is clean.",
)

Branch: main
Latest commit: 7117113 Create README_HW6 Controlled Agentic Workflow
Repository status: Working tree is clean.


In [4]:
os.chdir(PROJECT_ROOT)

print("Current working directory:")
print(Path.cwd())

Current working directory:
/content/health-psychology-rag-kb


In [5]:
!pip -q install langgraph

In [6]:
from langgraph.graph import StateGraph, START, END

print("LangGraph is ready.")

LangGraph is ready.


In [7]:
HW7_ROOT = PROJECT_ROOT / "HW7_langgraph_orchestration"

HW7_SCRIPTS = HW7_ROOT / "scripts"
HW7_OUTPUTS = HW7_ROOT / "outputs"


for folder in [
    HW7_ROOT,
    HW7_SCRIPTS,
    HW7_OUTPUTS,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


print("Homework 7 workspace:")
print(HW7_ROOT)

print("\nHomework 7 folders:")

for folder in [
    HW7_SCRIPTS,
    HW7_OUTPUTS,
]:
    print(
        "-",
        folder.relative_to(PROJECT_ROOT),
    )

Homework 7 workspace:
/content/health-psychology-rag-kb/HW7_langgraph_orchestration

Homework 7 folders:
- HW7_langgraph_orchestration/scripts
- HW7_langgraph_orchestration/outputs


In [8]:
print("Project:", REPO_NAME)
print("Project root:", PROJECT_ROOT)
print("Homework 7 root:", HW7_ROOT)
print("LangGraph import: OK")
print("Git branch:", current_branch)

git_status = run_git_command(
    ["status", "--short"]
)

print(
    "Git status:",
    git_status if git_status else "Working tree is clean.",
)

print("\nStage 1 completed successfully.")

Project: health-psychology-rag-kb
Project root: /content/health-psychology-rag-kb
Homework 7 root: /content/health-psychology-rag-kb/HW7_langgraph_orchestration
LangGraph import: OK
Git branch: main
Git status: Working tree is clean.

Stage 1 completed successfully.


## Stage 2 — Define the orchestration use case

The project already contains several capabilities:

- Health Psychology RAG for evidence-based questions;
- product usage analytics;
- role-based access control for restricted analytics;
- clarification handling for unsupported or unclear requests.


LangGraph will coordinate which capability should handle a user request and which execution path is allowed for the current authenticated user.

The workflow will make the following decisions explicit:

1. What type of request did the user make?
2. Should the request use Health Psychology retrieval, analytics, or clarification?
3. If analytics are requested, is the authenticated user allowed to access them?
4. Which nodes were executed?
5. What final result was produced?

A key scenario uses the same analytics request with two different trusted user roles.

For an authenticated admin:

```text
analytics request
→ check access
→ analytics tool
→ final answer

In [11]:
TEST_SCENARIOS = [
    {
        "name": "Health Psychology question",
        "user_request": "Why can't I work effectively with chronic back pain?",
        "user_role": "user",
        "expected_route": "health_psychology",
        "expected_access": None,
    },
    {
        "name": "Analytics request — admin",
        "user_request": "Show me product analytics for the last 7 days.",
        "user_role": "admin",
        "expected_route": "analytics",
        "expected_access": True,
    },
    {
        "name": "Analytics request — external user",
        "user_request": "Show me product analytics for the last 7 days.",
        "user_role": "user",
        "expected_route": "analytics",
        "expected_access": False,
    },
    {
        "name": "Unclear request",
        "user_request": "I feel tired and annoyed lately and I don't know why.",
        "user_role": "user",
        "expected_route": "clarification",
        "expected_access": None,
    },
]


print("Defined test scenarios:", len(TEST_SCENARIOS))

for index, scenario in enumerate(TEST_SCENARIOS, start=1):
    print(f"\nScenario {index}: {scenario['name']}")
    print("Request:", scenario["user_request"])
    print("Trusted role:", scenario["user_role"])
    print("Expected route:", scenario["expected_route"])
    print("Expected analytics access:", scenario["expected_access"])

Defined test scenarios: 4

Scenario 1: Health Psychology question
Request: Why can't I work effectively with chronic back pain?
Trusted role: user
Expected route: health_psychology
Expected analytics access: None

Scenario 2: Analytics request — admin
Request: Show me product analytics for the last 7 days.
Trusted role: admin
Expected route: analytics
Expected analytics access: True

Scenario 3: Analytics request — external user
Request: Show me product analytics for the last 7 days.
Trusted role: user
Expected route: analytics
Expected analytics access: False

Scenario 4: Unclear request
Request: I feel tired and annoyed lately and I don't know why.
Trusted role: user
Expected route: clarification
Expected analytics access: None


## Stage 3 — Define the shared LangGraph state

LangGraph passes a shared state object between workflow nodes.

For this workflow, the state stores:

- the user request;
- the trusted user role;
- the selected route;
- analytics access status;
- the latest tool result;
- the final answer;
- the list of executed nodes.

The state makes the workflow decisions and execution path explicit and inspectable.

In [12]:
from typing import Any, Literal, TypedDict


UserRole = Literal[
    "admin",
    "user",
]

Route = Literal[
    "health_psychology",
    "analytics",
    "clarification",
]


class WorkflowState(TypedDict, total=False):
    user_request: str
    user_role: UserRole

    route: Route
    analytics_access: bool | None

    tool_result: dict[str, Any] | None
    final_answer: str | None

    executed_nodes: list[str]


print("WorkflowState defined successfully.")

WorkflowState defined successfully.


In [13]:
example_state: WorkflowState = {
    "user_request": "Show me product analytics for the last 7 days.",
    "user_role": "admin",
    "route": None,
    "analytics_access": None,
    "tool_result": None,
    "final_answer": None,
    "executed_nodes": [],
}


print("Initial workflow state:")

for key, value in example_state.items():
    print(f"{key}: {value}")

Initial workflow state:
user_request: Show me product analytics for the last 7 days.
user_role: admin
route: None
analytics_access: None
tool_result: None
final_answer: None
executed_nodes: []


In [14]:
from typing import Any, Literal, TypedDict


UserRole = Literal[
    "admin",
    "user",
]

Route = Literal[
    "health_psychology",
    "analytics",
    "clarification",
]


class WorkflowState(TypedDict, total=False):
    user_request: str
    user_role: UserRole

    route: Route | None
    analytics_access: bool | None

    tool_result: dict[str, Any] | None
    final_answer: str | None

    executed_nodes: list[str]


print("WorkflowState defined successfully.")

WorkflowState defined successfully.


## Stage 4 — Prepare tools and helper functions

This stage prepares the simple functions used by the graph.

The underlying capabilities are intentionally lightweight because the focus of Homework 7 is orchestration rather than rebuilding previous homework components.

The workflow uses:

- a mock Health Psychology retrieval tool;
- a mock product analytics tool;
- a helper for recording executed graph nodes.

Access control is not implemented inside the analytics tool. It will be handled explicitly by the LangGraph workflow before the tool can be executed.

In [15]:
def append_node(
    state: WorkflowState,
    node_name: str,
) -> list[str]:
    return [
        *state.get("executed_nodes", []),
        node_name,
    ]


print("Execution trace helper is ready.")

Execution trace helper is ready.


In [16]:
def search_health_psychology(
    query: str,
) -> dict[str, Any]:
    """
    Mock interface representing the existing
    Health Psychology RAG capability.
    """

    return {
        "tool_name": "search_health_psychology",
        "success": True,
        "query": query,
        "result": (
            "Relevant Health Psychology context was retrieved "
            "about chronic pain, pain perception, psychological "
            "factors, and daily functioning."
        ),
    }


print("Health Psychology retrieval tool is ready.")

Health Psychology retrieval tool is ready.


In [17]:
def get_usage_analytics(
    period_days: int = 7,
) -> dict[str, Any]:
    """
    Mock interface representing the existing
    product analytics capability.
    """

    return {
        "tool_name": "get_usage_analytics",
        "success": True,
        "period_days": period_days,
        "total_users": 6,
        "new_users": 4,
        "returning_users": 2,
        "total_sessions": 9,
        "total_queries": 15,
    }


print("Product analytics tool is ready.")

Product analytics tool is ready.


In [18]:
rag_test = search_health_psychology(
    "Why can't I work effectively with chronic back pain?"
)

analytics_test = get_usage_analytics(
    period_days=7
)


print("RAG tool test:")
print(rag_test)

print("\nAnalytics tool test:")
print(analytics_test)

RAG tool test:
{'tool_name': 'search_health_psychology', 'success': True, 'query': "Why can't I work effectively with chronic back pain?", 'result': 'Relevant Health Psychology context was retrieved about chronic pain, pain perception, psychological factors, and daily functioning.'}

Analytics tool test:
{'tool_name': 'get_usage_analytics', 'success': True, 'period_days': 7, 'total_users': 6, 'new_users': 4, 'returning_users': 2, 'total_sessions': 9, 'total_queries': 15}


## Stage 5 — Implement the LangGraph nodes

Each LangGraph node reads the shared workflow state and returns a partial state update.

The workflow uses separate nodes for:

- request classification;
- Health Psychology retrieval;
- analytics access validation;
- analytics execution;
- access denial;
- clarification;
- final answer construction.

In [19]:
def classify_request_node(
    state: WorkflowState,
) -> dict[str, Any]:

    request = state["user_request"].lower()

    analytics_keywords = [
        "analytics",
        "users",
        "sessions",
        "queries",
        "usage",
    ]

    health_psychology_keywords = [
        "pain",
        "chronic",
        "stress",
        "psychology",
        "health",
        "behavior",
        "behaviour",
        "com-b",
    ]

    if any(
        keyword in request
        for keyword in analytics_keywords
    ):
        route = "analytics"

    elif any(
        keyword in request
        for keyword in health_psychology_keywords
    ):
        route = "health_psychology"

    else:
        route = "clarification"

    return {
        "route": route,
        "executed_nodes": append_node(
            state,
            "classify_request",
        ),
    }


print("classify_request node is ready.")

classify_request node is ready.


In [20]:
def retrieve_health_psychology_node(
    state: WorkflowState,
) -> dict[str, Any]:

    result = search_health_psychology(
        state["user_request"]
    )

    return {
        "tool_result": result,
        "executed_nodes": append_node(
            state,
            "retrieve_health_psychology",
        ),
    }


print("retrieve_health_psychology node is ready.")

retrieve_health_psychology node is ready.


In [21]:
def check_analytics_access_node(
    state: WorkflowState,
) -> dict[str, Any]:

    access_granted = (
        state.get("user_role") == "admin"
    )

    return {
        "analytics_access": access_granted,
        "executed_nodes": append_node(
            state,
            "check_analytics_access",
        ),
    }


print("check_analytics_access node is ready.")

check_analytics_access node is ready.


In [22]:
def get_usage_analytics_node(
    state: WorkflowState,
) -> dict[str, Any]:

    result = get_usage_analytics(
        period_days=7
    )

    return {
        "tool_result": result,
        "executed_nodes": append_node(
            state,
            "get_usage_analytics",
        ),
    }


print("get_usage_analytics node is ready.")

get_usage_analytics node is ready.


In [23]:
def deny_access_node(
    state: WorkflowState,
) -> dict[str, Any]:

    result = {
        "tool_name": "get_usage_analytics",
        "success": False,
        "error": "analytics_access_denied",
    }

    return {
        "tool_result": result,
        "executed_nodes": append_node(
            state,
            "deny_access",
        ),
    }


print("deny_access node is ready.")

deny_access node is ready.


In [24]:
def ask_clarification_node(
    state: WorkflowState,
) -> dict[str, Any]:

    result = {
        "success": True,
        "message": (
            "Please clarify whether your question is about "
            "Health Psychology content or product analytics."
        ),
    }

    return {
        "tool_result": result,
        "executed_nodes": append_node(
            state,
            "ask_clarification",
        ),
    }


print("ask_clarification node is ready.")

ask_clarification node is ready.


In [25]:
def build_answer_node(
    state: WorkflowState,
) -> dict[str, Any]:

    route = state.get("route")
    tool_result = state.get("tool_result")

    if route == "health_psychology":
        final_answer = (
            tool_result.get("result")
            if tool_result
            else "No Health Psychology context was retrieved."
        )

    elif route == "analytics":

        if state.get("analytics_access") is True:
            final_answer = (
                f"Product analytics for the last "
                f"{tool_result['period_days']} days: "
                f"{tool_result['total_users']} users, "
                f"{tool_result['total_sessions']} sessions, "
                f"and {tool_result['total_queries']} queries."
            )

        else:
            final_answer = (
                "Access denied. Product analytics are available "
                "only to authorized internal users."
            )

    else:
        final_answer = (
            tool_result.get("message")
            if tool_result
            else "Please clarify your request."
        )

    return {
        "final_answer": final_answer,
        "executed_nodes": append_node(
            state,
            "build_answer",
        ),
    }


print("build_answer node is ready.")

build_answer node is ready.


In [26]:
for scenario in TEST_SCENARIOS:

    initial_state: WorkflowState = {
        "user_request": scenario["user_request"],
        "user_role": scenario["user_role"],
        "executed_nodes": [],
    }

    result = classify_request_node(
        initial_state
    )

    print(
        scenario["name"],
        "→",
        result["route"],
    )

Health Psychology question → health_psychology
Analytics request — admin → analytics
Analytics request — external user → analytics
Unclear request → clarification


## Stage 6 — Build and compile the LangGraph workflow

This stage connects the workflow nodes into a LangGraph graph.

The graph uses two conditional routing decisions:

1. after request classification:
   - Health Psychology;
   - analytics;
   - clarification;

2. after the analytics access check:
   - authorized admin → analytics tool;
   - external user → access denied.

All routes eventually converge on the shared `build_answer` node.

In [27]:
def route_after_classification(
    state: WorkflowState,
) -> str:

    route = state.get("route")

    if route == "health_psychology":
        return "retrieve_health_psychology"

    if route == "analytics":
        return "check_analytics_access"

    return "ask_clarification"


print("Classification router is ready.")

Classification router is ready.


In [28]:
def route_after_access_check(
    state: WorkflowState,
) -> str:

    if state.get("analytics_access") is True:
        return "get_usage_analytics"

    return "deny_access"


print("Access router is ready.")

Access router is ready.


In [29]:
workflow = StateGraph(WorkflowState)


workflow.add_node(
    "classify_request",
    classify_request_node,
)

workflow.add_node(
    "retrieve_health_psychology",
    retrieve_health_psychology_node,
)

workflow.add_node(
    "check_analytics_access",
    check_analytics_access_node,
)

workflow.add_node(
    "get_usage_analytics",
    get_usage_analytics_node,
)

workflow.add_node(
    "deny_access",
    deny_access_node,
)

workflow.add_node(
    "ask_clarification",
    ask_clarification_node,
)

workflow.add_node(
    "build_answer",
    build_answer_node,
)


print("Graph nodes added successfully.")

Graph nodes added successfully.


In [30]:
# Entry point.

workflow.add_edge(
    START,
    "classify_request",
)


# First conditional decision:
# what type of request is this?

workflow.add_conditional_edges(
    "classify_request",
    route_after_classification,
    {
        "retrieve_health_psychology": "retrieve_health_psychology",
        "check_analytics_access": "check_analytics_access",
        "ask_clarification": "ask_clarification",
    },
)


# Health Psychology route.

workflow.add_edge(
    "retrieve_health_psychology",
    "build_answer",
)


# Analytics route:
# check trusted user access first.

workflow.add_conditional_edges(
    "check_analytics_access",
    route_after_access_check,
    {
        "get_usage_analytics": "get_usage_analytics",
        "deny_access": "deny_access",
    },
)


workflow.add_edge(
    "get_usage_analytics",
    "build_answer",
)

workflow.add_edge(
    "deny_access",
    "build_answer",
)


# Clarification route.

workflow.add_edge(
    "ask_clarification",
    "build_answer",
)


# Shared exit.

workflow.add_edge(
    "build_answer",
    END,
)


print("Graph edges added successfully.")

Graph edges added successfully.


In [31]:
app = workflow.compile()

print("LangGraph workflow compiled successfully.")

LangGraph workflow compiled successfully.


In [32]:
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify_request(classify_request)
	retrieve_health_psychology(retrieve_health_psychology)
	check_analytics_access(check_analytics_access)
	get_usage_analytics(get_usage_analytics)
	deny_access(deny_access)
	ask_clarification(ask_clarification)
	build_answer(build_answer)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify_request;
	ask_clarification --> build_answer;
	check_analytics_access -.-> deny_access;
	check_analytics_access -.-> get_usage_analytics;
	classify_request -.-> ask_clarification;
	classify_request -.-> check_analytics_access;
	classify_request -.-> retrieve_health_psychology;
	deny_access --> build_answer;
	get_usage_analytics --> build_answer;
	retrieve_health_psychology --> build_answer;
	build_answer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Stage 7 — Run the workflow through different execution paths

This stage runs the compiled LangGraph workflow using the four predefined test scenarios.

The goal is to verify that:

- Health Psychology questions use the retrieval path;
- admins can access product analytics;
- external users are denied analytics access;
- unclear requests use the clarification path.

The two analytics scenarios use the same user request but different trusted user roles, allowing us to compare their execution paths.

In [33]:
WORKFLOW_RESULTS = []


for scenario in TEST_SCENARIOS:

    initial_state: WorkflowState = {
        "user_request": scenario["user_request"],
        "user_role": scenario["user_role"],
        "route": None,
        "analytics_access": None,
        "tool_result": None,
        "final_answer": None,
        "executed_nodes": [],
    }

    final_state = app.invoke(initial_state)

    WORKFLOW_RESULTS.append(
        {
            "scenario": scenario,
            "final_state": final_state,
        }
    )


print(
    "Workflow scenarios executed:",
    len(WORKFLOW_RESULTS),
)

Workflow scenarios executed: 4


In [34]:
for index, item in enumerate(
    WORKFLOW_RESULTS,
    start=1,
):

    scenario = item["scenario"]
    state = item["final_state"]

    print("=" * 70)
    print(
        f"Scenario {index}: "
        f"{scenario['name']}"
    )

    print("\nRequest:")
    print(state["user_request"])

    print("\nTrusted role:")
    print(state["user_role"])

    print("\nSelected route:")
    print(state.get("route"))

    print("\nAnalytics access:")
    print(state.get("analytics_access"))

    print("\nExecuted nodes:")
    for node in state.get(
        "executed_nodes",
        [],
    ):
        print("-", node)

    print("\nTool result:")
    print(state.get("tool_result"))

    print("\nFinal answer:")
    print(state.get("final_answer"))

    print()

Scenario 1: Health Psychology question

Request:
Why can't I work effectively with chronic back pain?

Trusted role:
user

Selected route:
health_psychology

Analytics access:
None

Executed nodes:
- classify_request
- retrieve_health_psychology
- build_answer

Tool result:
{'tool_name': 'search_health_psychology', 'success': True, 'query': "Why can't I work effectively with chronic back pain?", 'result': 'Relevant Health Psychology context was retrieved about chronic pain, pain perception, psychological factors, and daily functioning.'}

Final answer:
Relevant Health Psychology context was retrieved about chronic pain, pain perception, psychological factors, and daily functioning.

Scenario 2: Analytics request — admin

Request:
Show me product analytics for the last 7 days.

Trusted role:
admin

Selected route:
analytics

Analytics access:
True

Executed nodes:
- classify_request
- check_analytics_access
- get_usage_analytics
- build_answer

Tool result:
{'tool_name': 'get_usage_anal

In [35]:
print("Workflow validation:\n")


for item in WORKFLOW_RESULTS:

    scenario = item["scenario"]
    state = item["final_state"]

    route_ok = (
        state.get("route")
        == scenario["expected_route"]
    )

    access_ok = (
        state.get("analytics_access")
        == scenario["expected_access"]
    )

    passed = route_ok and access_ok

    print(
        f"{scenario['name']}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

    print(
        "  route:",
        state.get("route"),
        "| expected:",
        scenario["expected_route"],
    )

    print(
        "  access:",
        state.get("analytics_access"),
        "| expected:",
        scenario["expected_access"],
    )

    print()

Workflow validation:

Health Psychology question: PASS
  route: health_psychology | expected: health_psychology
  access: None | expected: None

Analytics request — admin: PASS
  route: analytics | expected: analytics
  access: True | expected: True

Analytics request — external user: PASS
  route: analytics | expected: analytics
  access: False | expected: False

Unclear request: PASS
  route: clarification | expected: clarification
  access: None | expected: None



## Stage 8 — Inspect and compare execution traces

This stage compares the execution traces produced by LangGraph.

The most important comparison uses the same analytics request for two different trusted user roles.

The request itself does not determine access.

Instead, the workflow checks the trusted `user_role` stored in shared state and routes the request to either:

- the analytics tool; or
- the access-denied path.

This makes authorization decisions explicit and traceable in the graph.

In [36]:
admin_result = WORKFLOW_RESULTS[1]["final_state"]
user_result = WORKFLOW_RESULTS[2]["final_state"]


print("SAME USER REQUEST")
print("-" * 60)

print(admin_result["user_request"])


print("\nADMIN PATH")
print("-" * 60)

print("Trusted role:", admin_result["user_role"])
print("Analytics access:", admin_result["analytics_access"])
print(
    "Execution path:",
    " -> ".join(admin_result["executed_nodes"]),
)
print("Final answer:")
print(admin_result["final_answer"])


print("\nEXTERNAL USER PATH")
print("-" * 60)

print("Trusted role:", user_result["user_role"])
print("Analytics access:", user_result["analytics_access"])
print(
    "Execution path:",
    " -> ".join(user_result["executed_nodes"]),
)
print("Final answer:")
print(user_result["final_answer"])

SAME USER REQUEST
------------------------------------------------------------
Show me product analytics for the last 7 days.

ADMIN PATH
------------------------------------------------------------
Trusted role: admin
Analytics access: True
Execution path: classify_request -> check_analytics_access -> get_usage_analytics -> build_answer
Final answer:
Product analytics for the last 7 days: 6 users, 9 sessions, and 15 queries.

EXTERNAL USER PATH
------------------------------------------------------------
Trusted role: user
Analytics access: False
Execution path: classify_request -> check_analytics_access -> deny_access -> build_answer
Final answer:
Access denied. Product analytics are available only to authorized internal users.


In [37]:
print("LANGGRAPH EXECUTION PATHS\n")


for item in WORKFLOW_RESULTS:

    scenario = item["scenario"]
    state = item["final_state"]

    path = " -> ".join(
        state.get("executed_nodes", [])
    )

    print(scenario["name"])
    print("Route:", state.get("route"))
    print("Path:", path)
    print()

LANGGRAPH EXECUTION PATHS

Health Psychology question
Route: health_psychology
Path: classify_request -> retrieve_health_psychology -> build_answer

Analytics request — admin
Route: analytics
Path: classify_request -> check_analytics_access -> get_usage_analytics -> build_answer

Analytics request — external user
Route: analytics
Path: classify_request -> check_analytics_access -> deny_access -> build_answer

Unclear request
Route: clarification
Path: classify_request -> ask_clarification -> build_answer



### Execution trace observation

The graph produced four distinct execution paths.

The clearest example is the analytics workflow.

The admin and external user submitted the same request and received the same `analytics` route after classification.

However, the access-control node inspected trusted workflow state and produced different next steps:

```text
admin
→ get_usage_analytics

external user
→ deny_access

## Stage 9 — Compare custom orchestration with LangGraph

The same workflow could be implemented with regular Python `if/else` logic.

For a small workflow, the custom implementation would be shorter and would require less framework-specific code.

LangGraph adds explicit structure around:

- shared state;
- nodes;
- edges;
- conditional routing;
- execution paths.

In this homework, the main advantage is visibility.

The workflow clearly shows two different types of decisions:

```text
request intent
→ Health Psychology / analytics / clarification

trusted user state
→ analytics allowed / analytics denied

The admin and external user submit the same analytics request, but the graph routes them differently because user_role is part of shared state.

This makes the workflow easier to inspect, debug, and extend than a growing set of nested if/else conditions.

For this small example, LangGraph is not necessary from a technical perspective.

Its value becomes more important when the workflow grows and needs additional routes, tools, retries, human review, memory, or more complex state transitions.


### Main takeaway

LangGraph did not make the underlying tools smarter.

It made the orchestration more explicit.

The framework separates:

```text
what each step does
→ nodes

what happens next
→ edges

what the workflow remembers
→ state

how different conditions change the path
→ conditional routing
```

For a small workflow, custom Python remains simpler.

For a larger agentic system, the graph structure can make control flow easier to understand and maintain.